# Candidate selection and model-specific diagnostics

This notebook reproduces the two frozen Ridge candidate experiments on selection origins `2007Q4` through `2016Q3`. It runs temporal-validity tests, validates checkpoint and baseline lineage, reports point, residual, interval, and bootstrap diagnostics, and applies the predeclared promotion criteria. It does not read candidate confirmation or holdout results.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

DRIVE_PROJECT_ROOT = Path('/content/drive/MyDrive/ds_portfolio/project_02_coal_production_forecasting')
DATA_ROOT = DRIVE_PROJECT_ROOT / 'data'
RUNS_ROOT = DRIVE_PROJECT_ROOT / 'runs'

In [ ]:
import subprocess
import sys

REPO_URL = 'https://github.com/ahmaddshbg-blip/regional-coal-production-forecasting.git'
REPO_DIR = Path('/content/regional-coal-production-forecasting')
if (REPO_DIR / '.git').is_dir():
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-r', str(REPO_DIR / 'requirements-lock.txt')],
    check=True,
)
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', 'scikit-learn==1.9.1'],
    check=True,
)
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-e', str(REPO_DIR), '--no-deps'],
    check=True,
)
source_root = str(REPO_DIR / 'src')
if source_root not in sys.path:
    sys.path.insert(0, source_root)
revision = subprocess.run(
    ['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'],
    check=True, capture_output=True, text=True,
).stdout.strip()
print('Code revision:', revision)

In [ ]:
import os

os.chdir(REPO_DIR)
os.environ['PROJECT_DATA_ROOT'] = str(DATA_ROOT)
os.environ['PROJECT_RUNS_ROOT'] = str(RUNS_ROOT)
subprocess.run(
    [sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'],
    cwd=REPO_DIR, check=True,
)

## Frozen boundary

Both configurations stop at selection target `2017Q3`. The reusable runner rejects a panel containing later target values, writes `diagnostic_gate_status = awaiting_review`, and fixes both `confirmation_opened` and `holdout_opened` to `false`. Review decisions are recorded separately in `DEC-014` and `DEC-016`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from matplotlib.ticker import PercentFormatter

from coal_forecasting.candidate_selection import run_candidate_selection_evaluation

candidate_configs = {
    'v1 log change': REPO_DIR / 'configs' / 'candidate.json',
    'v2 raw delta': REPO_DIR / 'configs' / 'candidate_v2.json',
}
results = {
    label: run_candidate_selection_evaluation(
        REPO_DIR / 'configs' / 'project.json',
        config_path,
        root=REPO_DIR,
    )
    for label, config_path in candidate_configs.items()
}
for label, result in results.items():
    print(label, result['manifest']['run_id'], 'alpha =', result['selected_alpha'])

## Lineage and safety checks

In [ ]:
manifest_rows = []
for label, result in results.items():
    manifest = result['manifest']
    manifest_rows.append(
        {
            'candidate': label,
            'run_id': manifest['run_id'],
            'procedure_id': manifest['procedure_id'],
            'source_data_run': manifest['source_data_run'],
            'source_baseline_run': manifest['source_baseline_run'],
            'selected_alpha': manifest['parameters']['selected_alpha'],
            'diagnostic_gate': manifest['checks']['diagnostic_gate_status'],
            'confirmation_opened': manifest['checks']['confirmation_opened'],
            'holdout_opened': manifest['checks']['holdout_opened'],
            'git_commit': manifest['git']['commit'],
            'dirty_worktree': manifest['git']['dirty'],
        }
    )
display(pd.DataFrame.from_records(manifest_rows))

## Alpha selection and point forecast evidence

In [ ]:
for label, result in results.items():
    print(label)
    display(result['alpha_summary'])
    display(
        result['candidate_metrics'][
            ['horizon', 'wape', 'aggregate_signed_bias', 'median_state_mase', 'prediction_coverage']
        ]
    )
    display(result['bootstrap_skill'])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
colors = {'v1 log change': '#b91c1c', 'v2 raw delta': '#2563eb'}
for label, result in results.items():
    alpha = result['alpha_summary']
    axes[0].plot(alpha['alpha'], alpha['mean_wape_skill'], marker='o', label=label, color=colors[label])
    skill = result['bootstrap_skill'].dropna(subset=['horizon'])
    axes[1].errorbar(
        skill['horizon'] + (-0.04 if label.startswith('v1') else 0.04),
        skill['point_wape_skill'],
        yerr=[skill['point_wape_skill'] - skill['lower_95'], skill['upper_95'] - skill['point_wape_skill']],
        fmt='o-', capsize=3, label=label, color=colors[label],
    )
axes[0].axhline(0, color='#111827', linewidth=1)
axes[0].set_xscale('log')
axes[0].set_title('Selection objective by alpha')
axes[0].set_xlabel('Ridge alpha')
axes[0].set_ylabel('Mean horizon WAPE skill')
axes[1].axhline(0, color='#111827', linewidth=1)
axes[1].set_xticks([1, 2, 3, 4])
axes[1].set_title('Paired block-bootstrap skill')
axes[1].set_xlabel('Forecast horizon')
axes[1].set_ylabel('WAPE skill with 95% interval')
for axis in axes:
    axis.yaxis.set_major_formatter(PercentFormatter(1.0))
    axis.grid(axis='y', alpha=0.25)
    axis.legend(frameon=False)
fig.tight_layout()

## Model-working and interval diagnostics

In [ ]:
for label, result in results.items():
    print(label)
    display(result['diagnostic_summary'])
    autocorrelation = result['diagnostic_autocorrelation'].groupby(
        ['horizon', 'lag'], as_index=False
    ).agg(
        eligible_states=('correlation', 'count'),
        median_correlation=('correlation', 'median'),
        mean_absolute_correlation=('correlation', lambda values: values.abs().mean()),
    )
    display(autocorrelation)
    display(result['interval_metrics'])

## Frozen promotion criteria

In [ ]:
promotion_rows = []
for label, result in results.items():
    candidate = result['candidate_metrics'][['horizon', 'wape', 'median_state_mase']].copy()
    baseline = result['baseline_metrics']
    primary = []
    for horizon in [1, 2, 3, 4]:
        comparator = 'seasonal_naive' if horizon == 3 else 'persistence'
        value = baseline.loc[
            baseline['horizon'].eq(horizon) & baseline['model'].eq(comparator), 'wape'
        ].iloc[0]
        primary.append(value)
    candidate['comparator_wape'] = primary
    candidate['wape_skill'] = 1 - candidate['wape'] / candidate['comparator_wape']
    secondary = baseline.loc[baseline['model'].eq('persistence'), 'median_state_mase'].mean()
    mean_skill = candidate['wape_skill'].mean()
    promotion_rows.append(
        {
            'candidate': label,
            'coverage_100_percent': bool(result['candidate_metrics']['prediction_coverage'].eq(1.0).all()),
            'mean_wape_skill': mean_skill,
            'mean_skill_at_least_5_percent': mean_skill >= 0.05,
            'positive_skill_horizons': int(candidate['wape_skill'].gt(0).sum()),
            'worst_horizon_skill': candidate['wape_skill'].min(),
            'mean_median_state_mase': candidate['median_state_mase'].mean(),
            'secondary_comparator': secondary,
            'state_mase_no_worse': candidate['median_state_mase'].mean() <= secondary,
            'confirmation_opened': result['manifest']['checks']['confirmation_opened'],
            'holdout_opened': result['manifest']['checks']['holdout_opened'],
        }
    )
promotion = pd.DataFrame.from_records(promotion_rows)
display(promotion.style.format({
    'mean_wape_skill': '{:.2%}',
    'worst_horizon_skill': '{:.2%}',
    'mean_median_state_mase': '{:.3f}',
    'secondary_comparator': '{:.3f}',
}))

## Decision boundary

V1 is rejected by `DEC-014` for severe negative point skill and raw-scale underforecasting. V2 is rejected by `DEC-016`: it returns close to persistence and calibrates intervals adequately, but does not reach the 5 percent materiality threshold, does not establish robust mean skill, and worsens the state-balanced criterion. The appropriate frozen baseline is retained. This notebook does not authorize confirmation, holdout evaluation, or a third candidate.